# Chapter 10 Companion Notebook: Tree-Based Models: Random Forest Classification

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch10_Tree_Based_Models_Random_Forest.ipynb)

This notebook accompanies Chapter 10 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
- Click "Upload" and select this file and the data file.

# Bank customers: Random forest

### Use "Bank customer.csv"
- age (numeric)  
- marital: marital status (categorical: "married", "divorced", "single"; "divorced" means divorced or widowed)  - education (categorical: "secondary", "primary", "tertiary")  
- default: has credit in default? (binary: "yes", "no")  
- balance: average yearly balance, in euros (numeric)  
- housing: has housing loan? (binary: "yes", "no")  
- loan: has personal loan? (binary: "yes", "no")  
- duration: last contact duration, in seconds (numeric)  
- campaign: number of contacts performed during this campaign (numeric, includes last contact)  
- pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric)
- previous: number of contacts performed before this campaign (numeric)  
- poutcome: outcome of the previous marketing campaign (categorical: "failure", "success")
- deposit: has the client subscribed a term deposit? (binary: "yes", "no")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
# Read the data

df = pd.read_csv('Bank customer.csv')
df.head()

In [ ]:
# Create dummy variables for categorical columns

df = pd.get_dummies(df, drop_first=True)
df.head()

In [ ]:
# Define x and y. Split into train, test data

y=df.deposit_yes
x=df.drop('deposit_yes', axis=1)
xtrain, xtest, ytrain, ytest = train_test_split(x, y, random_state=1)

## Random forests

In [ ]:
#  Define and fit a decision tree model on the data

m1 = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=1).fit(xtrain, ytrain)

- RandomForestClassifier()
  - `n_estimators=100`: It specifies the number of decision trees that will be included in the forest.
  - `max_depth=5`: It sets the maximum depth of each decision tree in the forest.
  - `fit(xtrain, ytrain)`: Fits the Random Forest classifier with the training data.

In [ ]:
# Predict the target in the test data and display confusion matrix

pred1 = m1.predict(xtest)
ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(ytest, pred1)).plot()

- `m1.predict(xtest)`: It uses the trained Random Forest classifier (m1) to make predictions on the test dataset (xtest).
- `confusion_matrix(ytest, pred1)`: The confusion matrix shows the true positive, true negative, false positive, and false negative counts for a set of predictions compared to the true labels (ytest).
- `ConfusionMatrixDisplay()`: creaties a visual representation of a confusion matrix.

In [ ]:
print('Accuracy:', metrics.accuracy_score(ytest, pred1))

- `metric.accuracy_score`: calculates the accuracy of the predicted compared to the actual target values (ytest).

In [ ]:
# Classification report

print(classification_report(ytest, pred1))

- `classification report`: summarizies various metrics such as precision, recall, F1-score, and support for each class based on the predictions (pred) and actual labels (ytest).
- precision: the ratio of true positive predictions to the total number of positive predictions.
- recall: the ratio of true positive predictions to the total number of actual positive instances
- F1-score: combines both precision and recall into a single value, providing a balance between the two metrics for evaluating classification model performance.
- support: the number of occurrences of a specific class.

### Visualization

In [ ]:
# Display the decision trees from the random forest

from sklearn.tree import plot_tree
for i, tree in enumerate(m1.estimators_[:3]):  # Display the first three trees as an example
    plt.figure(figsize=(20, 10))
    plot_tree(tree, feature_names=x.columns, class_names=['No', 'Yes'], filled=True, fontsize=8)
    plt.title(f"Decision Tree {i+1}")

### Hyperparameter tuning

In [ ]:
# Use random search to find the best hyperparameters (It will take some time)

m2 = RandomForestClassifier()
param = {'n_estimators': randint(50,200), 'max_depth': randint(5,20)}
search=RandomizedSearchCV(m2, param_distributions = param, n_iter=5, cv=5).fit(xtrain, ytrain)

- `param = {}` is a dictionary with two hyperparameters to tune
    - `randint()` generate random integers within a specified range. Any integer between 50 (inclusive) and 200 (exclusive) could be generated.
    - `n_estimators` and `max_depth`: The values are randomly sampled from a uniform distribution from the range specified in `randint()`
    - The possible number of combinations is 150*15=2250 (50-199 for n_estimators, 5-19 for max_depth).
- `RandomizedSearchCV(m, param_distributions = param, n_iter=5, cv=5)`
    - `RandomizedSearchCV` is a technique for hyperparameter optimization
        - Unlike grid search, which exhaustively tries all possible combinations of hyperparameters, randomized search takes a random sampling approach. It specifies a range or distribution for each hyperparameter and then randomly samples a set of hyperparameter combinations to evaluate.
        - Instead of specifying specific values for hyperparameters, you define distributions or ranges for the hyperparameters. The random search then samples from these distributions to create different combinations.
    - `n_iter`: specifies how many different combinations of hyperparameters should be tried during the randomized search.
        - The randomized search algorithm will randomly sample `n_iter`(=5 in this case) different sets of hyperparameters from the specified parameter distributions.
        - After `n_iter` iterations, the algorithm selects the best set of hyperparameters based on the specified performance metric.
    - `cv`: the number of cross-validation folds. It evaluates the model's performance using cross-validation for each set of hyperparameters.
    - `param_distributions`: specifies the range of values for each hyperparameter that will be considered during the search.

In [ ]:
# Create a variable for the best model

best = search.best_estimator_
print('Best hyperparameters:',  search.best_params_)

- `search.best_estimator_`: extracts the best estimator from the search object.

In [ ]:
pred2 = best.predict(xtest)
ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(ytest, pred2)).plot();

In [ ]:
# Classification report

print(classification_report(ytest, pred2))

- Accuracy increased from 88% to 90%
- F1 increased from 5% to 47%.

In [ ]:
# Create a series containing feature importances from the model and feature names from the training data

imp = pd.Series(best.feature_importances_, index=xtrain.columns).sort_values(ascending=False)
imp.plot.bar()

- `pd.Series(best.feature_importances_, index=xtrain.columns)`: calculates the feature importances
    - `best.feature_importances_` provides an array of feature importances from the model that performed the best during the search.
    - xtrain.columns provides the names of the features.
    - The importances are combined into a pandas Series, where the feature names are used as the index.
- `sort_values(ascending=False)`: sorts the features in descending order of importance.
- `imp.plot.bar()`: generates a bar plot to visualize the sorted feature importances.

- Duration has the highest impact on the target variable.